In [2]:
import pandas as pd
import numpy as np
import os

data = pd.read_csv('retail_store_sales.csv')

df = pd.DataFrame(data)
df

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False
...,...,...,...,...,...,...,...,...,...,...,...
12570,TXN_9347481,CUST_18,Patisserie,Item_23_PAT,38.0,4.0,152.0,Credit Card,In-store,2023-09-03,NaN
12571,TXN_4009414,CUST_03,Beverages,Item_2_BEV,6.5,9.0,58.5,Cash,Online,2022-08-12,False
12572,TXN_5306010,CUST_11,Butchers,Item_7_BUT,14.0,10.0,140.0,Cash,Online,2024-08-24,NaN
12573,TXN_5167298,CUST_04,Furniture,Item_7_FUR,14.0,6.0,84.0,Cash,Online,2023-12-30,True


In [37]:
df.isna().sum()

Transaction_ID      0
Customer_ID         0
Category            0
Item                0
Price_Per_Unit      0
Quantity            0
Total_Spent         0
Payment_Method      0
Location            0
Transaction_Date    0
Discount_Applied    0
dtype: int64

In [10]:
df = df.rename(columns={
    'Transaction ID': 'Transaction_ID',
    'Customer ID' : 'Customer_ID',
    'Total Spent': 'Total_Spent',
    'Transaction Date': 'Transaction_Date',
    'Price Per Unit': 'Price_Per_Unit',
    'Payment Method': 'Payment_Method',
    'Discount Applied' : 'Discount_Applied'
    
})



In [31]:
# حدت القيم التي معاها نفس القيمه
price_item_groups = (
    df[df["Item"].notna()]
    .groupby("Price_Per_Unit")["Item"]
    .unique()
)
price_item_groups

Price_Per_Unit
5.0     [Item_1_FOOD, Item_1_PAT, Item_1_MILK, Item_1_...
6.5     [Item_2_FOOD, Item_2_BUT, Item_2_CEA, Item_2_F...
8.0     [Item_3_BUT, Item_3_FOOD, Item_3_MILK, Item_3_...
9.5     [Item_4_EHE, Item_4_BEV, Item_4_FOOD, Item_4_P...
11.0    [Item_5_CEA, Item_5_FUR, Item_5_FOOD, Item_5_E...
12.5    [Item_6_FOOD, Item_6_PAT, Item_6_MILK, Item_6_...
14.0    [Item_7_BEV, Item_7_FUR, Item_7_PAT, Item_7_FO...
15.5    [Item_8_BEV, Item_8_MILK, Item_8_BUT, Item_8_C...
17.0    [Item_9_FOOD, Item_9_CEA, Item_9_MILK, Item_9_...
18.5    [Item_10_PAT, Item_10_FOOD, Item_10_FUR, Item_...
20.0    [Item_11_FOOD, Item_11_BEV, Item_11_PAT, Item_...
21.5    [Item_12_BUT, Item_12_FOOD, Item_12_PAT, Item_...
23.0    [Item_13_EHE, Item_13_CEA, Item_13_BUT, Item_1...
24.5    [Item_14_FUR, Item_14_FOOD, Item_14_PAT, Item_...
26.0    [Item_15_CEA, Item_15_MILK, Item_15_FUR, Item_...
27.5    [Item_16_BEV, Item_16_FUR, Item_16_MILK, Item_...
29.0    [Item_17_MILK, Item_17_PAT, Item_17_FUR, Item_...

In [32]:
for price, items in price_item_groups.items():
    
    mask = (df["Item"].isna()) & (df["Price_Per_Unit"] == price)
    
    if len(items) > 1:
        df.loc[mask, "Item"] = np.random.choice(
            items,
            size=mask.sum()
        )
    else:
        df.loc[mask, "Item"] = items[0]

In [30]:
df['Quantity'].value_counts()

Quantity
10.0    1232
5.0     1228
7.0     1227
8.0     1226
3.0     1224
6.0     1220
2.0     1164
4.0     1155
9.0     1148
1.0     1147
Name: count, dtype: int64

In [22]:
# حساب النسب الحالية
Discount_distribution = df["Discount_Applied"].value_counts(normalize=True)

Discount_distribution

Discount_Applied
True     0.503701
False    0.496299
Name: proportion, dtype: float64

In [24]:
# تعبئة القيم المفقودة بنفس النسب
df.loc[df["Discount_Applied"].isna(), "Discount_Applied"] = np.random.choice(
    Discount_distribution.index,
    size=df["Discount_Applied"].isna().sum(),
    p=Discount_distribution.values
)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(8)
memory usage: 1.1+ MB


In [7]:
df['Item'].value_counts()

Item
Item_2_BEV      126
Item_25_FUR     113
Item_11_FUR     110
Item_16_MILK    109
Item_1_MILK     109
               ... 
Item_5_BEV        7
Item_13_BEV       7
Item_13_FUR       7
Item_21_PAT       6
Item_3_EHE        5
Name: count, Length: 200, dtype: int64

In [18]:
df.loc[df["Price_Per_Unit"].isna(), "Price_Per_Unit"] = (
    df["Total_Spent"] / df["Quantity"]
)

In [19]:
df.loc[df["Quantity"].isna(), "Quantity"] = (
    df["Total_Spent"] / df["Price_Per_Unit"]
)

In [20]:
df.loc[df["Total_Spent"].isna(), "Total_Spent"] = (
    df["Quantity"] * df["Price_Per_Unit"]
)

In [35]:
df = df.dropna(subset=["Total_Spent"])

In [36]:
df = df.dropna(subset=["Quantity"])